In [22]:
# main.py
import sys
import os
import re
import string
import numpy as np
import pandas as pd
from pathlib import Path
from arcgis.gis import GIS
from arcgis.geocoding import geocode, batch_geocode
import time
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt

import matplotlib.dates as mdates


# Add the parent directory of your module to sys.path
module_path = "/Users/joel/PyProjects/police-journal"
sys.path.insert(0, module_path)

from pdf_processor import GranbyPDFProcessor

In [23]:
pd.__version__

'2.3.1'

In [4]:
plt.rcParams['font.family'] = ['Helvetica Neue', 'Arial', 'DejaVu Sans' ]
pd.set_option('display.max_rows', 25)

date_format = mdates.DateFormatter('%b %Y')

In [5]:
geopath = "/Users/Shared/Granby_Police_Journals/geocoding"
geofile = os.path.join(geopath, "addresses_with_zips_geocodio_c82e6800d9ad78f18ce0c13b17071865a810bf95.csv")

# Initialize processor
processor = GranbyPDFProcessor(pdf_directory="/Users/Shared/Granby_Police_Journals", verbose=False)

In [7]:
# Process all PDFs and create DataFrame
df = processor.process_all_pdfs()


2025-12-22 13:38:55,713 - INFO - Found 156 PDF files to process
2025-12-22 13:38:57,325 - INFO - Created DataFrame with 6667 total records


In [8]:
check911 = ((df['Call Type'].str.contains('911') & (df['Primary Officer'].str.len() > 1))).astype('int')
df['Call Type'] = df['Call Type'].where(check911 == 0, '911 RESPONSE')

## Geocoding
### Address cleanup

In [9]:
# clean addresses

def address_only(txt):
    
    txt = txt.upper().strip()
    txt = " ".join(txt.split())
    txt = txt.replace(" ,", ",")
    txt = txt.replace("(CATI)", "")
    txt = txt.replace(", EAST, GRANBY", ", EAST GRANBY")
    txt = txt.replace("ROUTE 10 TIRE", "ROUTE TEN TIRE")
    txt = " ".join(txt.split())
    txt = txt.replace(",,", ",")
    txt = txt.replace(", ,", ",")
    txt = txt.replace(" ,", ",")
    txt = " ".join(txt.split())
    txt = txt.replace(", RD, ", " RD, ")
    txt = txt.replace(", DR, ", " DR, ")
    txt = txt.replace("SOUTHWICK, SOUTHWICK", ", SOUTHWICK")
    txt = txt.replace("17R EAST GRANBY RD, 17R EAST GRANBY RD,", "17R EAST GRANBY RD,")
    txt = txt.replace("375R NORTH GRANBY RD, 375R NORTH GRANBY RD,", "375R NORTH GRANBY RD,")
    txt = txt.replace("125 LOST ACRES RD, 150 LOST ACRES RD,", "150 LOST ACRES RD,")
    txt = txt.replace("AT STATE, LINE", "AT STATE, LINE")
    txt = txt.replace("SOUTH CONGRAGATIONAL, CHURCH", "242 Salmon Brook St, Granby").upper()
    
    
    txt = txt.replace("WINDSOR, LOCKS", "WINDSOR LOCKS")
    if txt.endswith(("33 TURKEY HILLS ROAD", "SEYMOUR ROAD Apt #: 6F")):
        txt = txt + ", EAST GRANBY"

    if txt.endswith("CONNECTICUT VALLEY COMMONS, LLC"):
        txt = txt + ", 19 HARTFORD AVE, GRANBY"

    placename = ''
    address = ''
    ambulances = ('SIMS', '165MED', 'LAFD-T', 'I4-BLS', '47MED', 'LA6', 'SIMSAL', 'GA14', 'LA3', '6F', 'SUFF', 'CANT')

    amb = re.search(r"(?P<begin>.*)(?P<amb>, (SIMS|165MED|LAFD-T|I4-BLS|47MED|LA6|SIMSAL|GA14|LA3|6F|SUFF|CANT|MORELLI, LORRAINE|SAYLOR, LAUREN)$)", txt)
    try:
        txt = amb.group("begin").strip()
    except:
        pass
    
    # append granby if missing
    if txt.endswith((' RD', ' ROAD', ' DRIVE', ' AV', ' ST', ' PLACE', ' LANE', ' STREET')):
        txt = txt.strip() + ", GRANBY"
        
    # remove apartment numbers
    apt = re.search(r"(?P<begin>.*)(?P<apt>APT.*)(?P<granby>, (GRANBY|SIMSBURY|EAST GRANBY|BLOOMFIELD|WINDSOR LOCKS).*)", txt)
    if apt:
        txt = apt.group("begin").strip() + apt.group("granby")

    # separate placename as text preceding first digit
    txt = txt[::-1]
        
    substr = re.match(r"(?P<address>.+\d)?(?P<placename>.*)", txt)
    if substr:
        if substr.group('placename'):
            placename = substr.group('placename')[::-1].strip()
        if substr.group('address'):
            address = substr.group('address')[::-1].strip()
    else:
        address = txt[::-1]

    if address.strip() == "":
        address = placename
        placename = ""

    placename = placename.strip(",")
        
    address = address.replace(",,", ",")
    address = address.replace(", ,", ",")
    address = address.replace(" ,", ",")
    address = address.replace(" 'S","")
    address = address.replace("MURTHA'S WAY - DUPLEX, CONSTRUCTION, ", "")
    address = address.replace("219 LOOMIS ST, 218 LOOMIS ST,", "219 LOOMIS ST,")
    address = address.replace("280 APARTMENTS, ", "")
    address = address.replace("8 THRONEBROOK RD, 8 THRONEBROOK RD,","8 THRONEBROOK RD,")
    address = address.replace("66, 150","150")
    address = address.replace("HARTOLAND","HARTLAND")
    address = address.replace(" NEAR NEW CELL, TOWER SITE","")
    address = address.replace("STRAWBERRYFIELDS", "STRAWBERRY FIELDS")
    
    if address.endswith(("33 TURKEY HILLS ROAD", "SEYMOUR ROAD APT #: 6F")):
        address = address + ", EAST GRANBY"

    amb = re.search(r"(?P<begin>.*)(?P<amb> (SIMS|165MED|LAFD-T|I4-BLS|47MED|LA6|SIMSAL|GA14|LA3|6F|SUFF|CANT|MORELLI, LORRAINE|SAYLOR, LAUREN)$)", address)
    try:
        address = amb.group("begin").strip()
    except:
        pass

    if address.count(",") > 1:
        address = address.replace(",", "", 1)
        address = " ".join(address.split())
        
    if address.count(",") < 1:
        if address in ["NORTH HOLLOW/ GRANVILLE", "PHELPS/QUARRY", "UPPER MEADOW"]:
            address = address + ", GRANBY"
    
    address = address.replace('/',' @ ')
    address = " ".join(address.split())
    
    return [placename, address]

In [10]:
def get_state(x):

    townlist = ['GRANBY', 'EAST GRANBY', 'SIMSBURY', 'HARTLAND', 'SUFFIELD', 'BARKHAMSTED', 'WINDSOR LOCKS', 'HARTFORD', 'WINDSOR', 'BLOOMFIELD', 'CANTON', 'NEWINGTON',
                'TORRINGTON', 'EAST HARTLAND', 'MANCHESTER', 'GREENWICH', 'WINSTED', 'TARIFFVILLE', 'AVON', 'BURLINGTON', 'NEW BRITIAN', 'BRISTOL', 'PLAINVILLE', 'VERNON']
    malist = ['SOUTHWICK', 'GRANVILLE', 'CHICOPEE',]
    
    if x in townlist:
        return 'CT'
    elif x in malist:
        return 'MA'
    else:
        return ''


In [11]:
def extract_street(x):
    
    if x.count(",") > 0:
        street = x.split(",")[0]
        street = street.strip()
        return street
    return x

def extract_streetname(x):
    
    if x.count(",") > 0:
        street = x.split(",")[0]
        street = re.sub(r'\d', '', street)
        street = street.strip()
        return street
    return x

In [12]:
address_only('SHIGLEY VILLAGE, 53 SOUTH MAIN ST Apt #: 11B, EAST granby')

['SHIGLEY VILLAGE', '53 SOUTH MAIN ST, EAST GRANBY']

In [13]:
df['Place Name'] = df['Location'].apply(lambda x: address_only(x)[0])
df['Address'] = df['Location'].apply(lambda x: address_only(x)[1])
df['Town'] = df['Address'].apply(lambda x: x.split(",")[::-1][0].strip())
df['State'] = df['Town'].apply(lambda x: get_state(x))
df['Street'] = df['Address'].apply(lambda x: extract_street(x))
df['Street Name'] = df['Address'].apply(lambda x: extract_streetname(x))

In [14]:
df.head(2)

,CFS #,Dispatch Type,Call Type,Dispatch Date,Dispatch Time,Location,Unit Id,Primary Officer,File Date,Dispatch Timestamp,Dispatch Hour,Call Category,Place Name,Address,Town,State,Street,Street Name
0,2500007968,17,BUSINESS CHECK,2025-07-14,00:12,"STOP & SHOP, 120 SALMON BROOK ST, GRANBY",260,"KUPCHIK, ADDISON",07-14-2025,2025-07-14 00:12:00,0,Patrol,STOP & SHOP,"120 SALMON BROOK ST, GRANBY",GRANBY,CT,120 SALMON BROOK ST,SALMON BROOK ST
1,2500007969,17,BUSINESS CHECK,2025-07-14,00:17,"ARROW CONCRETE, 560 SALMON BROOK ST, GRANBY",266,"DELOY, TYSON",07-14-2025,2025-07-14 00:17:00,0,Patrol,ARROW CONCRETE,"560 SALMON BROOK ST, GRANBY",GRANBY,CT,560 SALMON BROOK ST,SALMON BROOK ST


In [15]:
# Use geocodio geocoding 
geo = pd.read_csv(geofile)
geo = geo[geo['Accuracy Type'] != 'state']
geo['Accuracy Type'].value_counts()

Accuracy Type
rooftop                  1129
street_center             221
range_interpolation       185
intersection              167
nearest_rooftop_match      12
place                       8
Name: count, dtype: int64

In [16]:
# add geocodeio
df2 = df.merge(geo[['Street', 'Town', 'State', 'Postal Code', 'Latitude', 'Longitude', 'Accuracy Type', 'Accuracy Score']],
               on=['Street', 'Town', 'State'], how='left')

df2['ZIP Code'] = df2['Postal Code'].fillna(0).astype(int).astype(str).str.zfill(5)

# override middle/high school geocoding
df2['Latitude'] = df2['Latitude'].where(df2['Latitude'] != 41.960488, 41.960124921625656)
df2['Longitude'] = df2['Longitude'].where(df2['Longitude'] != -72.797766, -72.79283222814749)

# west granby post office
df2['Latitude'] = df2['Latitude'].where(df2['Address'] != 'UNITED STATED POST OFFICE -, WEST', 41.956512355376255)
df2['Longitude'] = df2['Longitude'].where(df2['Address'] != 'UNITED STATED POST OFFICE -, WEST', 41.956512355376255)

In [17]:
# geoapify geocoding
geos = []
for fileidx in range(1,5):
    geos.append(pd.read_csv(f"/Users/Shared/Granby_Police_Journals/geocoding/geocoded_by_geoapify-12_21_2025_{fileidx}.csv"))

geocodes = pd.concat(geos).rename(columns={'lat': 'Latitude', 'lon': 'Longitude', 'original_state': 'State'})
geocodes['Address'] = geocodes['original_location'] + ", " + geocodes["original_town"]
geocodes['postcode'] = geocodes['postcode'].fillna(0).astype(int).astype(str).str.zfill(5)

print(geocodes.shape)

# remove "center of town" coding
geocodes = geocodes[(geocodes.Longitude != -72.789313) & (geocodes.Latitude != 41.953830)]
print(geocodes.shape)

(1723, 27)
(1575, 27)


In [18]:
#add geocodio to missing
df3 = df2[df2['Latitude'].isna()].drop(columns=['Postal Code', 'Latitude', 'Longitude', 'Accuracy Type', 'Accuracy Score'])\
                         .merge(geocodes[['Address', 'State', 'postcode', 'Latitude', 'Longitude',
                                        'confidence', 'confidence_street_level','confidence_building_level']],
                                how='left', on=['Address', 'State']).rename(columns={'postcode': 'Postal Code'})

df4 = df3[df3['Latitude'].isna()]
df3 = df3[df3['Latitude'].notna()]

In [19]:
df4.shape, df3.shape

((309, 25), (124, 25))

In [20]:
g = df3[df3.confidence_street_level < .9][['Address', 'Street', 'Town', 'State']].drop_duplicates()
g['intersection'] = g.Street + ", " + g.Town + " " + g.State
# 41.950467, -72.785162

In [24]:
# Connect to ArcGIS Online anonymously
# A named user connection might be required for production use
gis = GIS() 

georesults = []
#for intersection_query in intersections['address'].tolist():
for intersection_query in g['intersection'].tolist():

    time.sleep(2)
    # Perform geocoding
    print(f"{intersection_query}")
    try:
        results = geocode(address=intersection_query, as_featureset=True)
        if results:
            geodf = results.sdf
            geodf['intersection'] = intersection_query
            georesults.append(geodf)
        else:
            print(f"      --->{intersection_query}: No intersection found.")
    except Exception as e:
        print(f"An error occurred: {e}")

NOTCH RD @ HUNGARY RD, GRANBY CT
SALMON BROOK ST @ EAST GRANBY RD, GRANBY CT
SALMON BROOK ST @ HARTFORD AVE, GRANBY CT
NORTH GRANBY RD @ MOUNTAIN RD, GRANBY CT
SILVER ST @ SILVER BROOK LN, GRANBY CT
GRANBROOK PARK RD @ HARTFORD AVE, EAST GRANBY CT
SALMON BROOK ST @ NORTH GRANBY RD, GRANBY CT
EAST GRANBY RD @ CANAL RD, GRANBY CT
HARTLAND BLVD @ MOOSEHORN RD, HARTLAND CT
NEWGATE RD @ OLD RD, EAST GRANBY CT
HUNGARY RD @ WESTVIEW DR, GRANBY CT
EAST GRANBY RD @ SALMON BROOK ST, GRANBY CT
W GRANBY RD @ N GRANBY RD, GRANBY CT
CANAL RD @ EAST GRANBY RD, GRANBY CT
SIMSBURY RD @ WEST GRANBY RD, GRANBY CT
MEADOW BROOK RD @ SALMON BROOK ST, GRANBY CT
NORTH GRANBY RD @ EAST ST, GRANBY CT
HARTFORD AVE @ OLD HARTFORD AVE, EAST GRANBY CT
MOUNTAIN RD @ NORTH GRANBY RD, GRANBY CT
COPPER BROOK CIR @ SALMON BROOK ST, GRANBY CT
WEST GRANBY RD @ STRONG RD, GRANBY CT
HARTFORD AVE @ SALMON BROOK ST, GRANBY CT
GRANVILLE RD @ SILVER ST, GRANBY CT
MECHANICSVILLE RD @ SALMON BROOK ST, GRANBY CT
SALMON BROOK ST @ 

In [78]:
rdf = pd.concat(georesults).reset_index(drop=False, names="ReturnNo")
rdf.head()

,ReturnNo,Loc_name,Status,Score,Match_addr,LongLabel,ShortLabel,Addr_type,Type,PlaceName,...,PotentialID,StrucType,StrucDet,StrucType1,StrucType2,StrucDet1,StrucDet2,OBJECTID,SHAPE,intersection
0,0,World,M,100.0,"Notch Rd & Hungary Rd, Granby, Connecticut, 06035","Notch Rd & Hungary Rd, Granby, CT, 06035, USA",Notch Rd & Hungary Rd,StreetInt,,,...,,,,,,,,1,"{""x"": -72.77906598187, ""y"": 41.976515991602, ""...","NOTCH RD @ HUNGARY RD, GRANBY CT"
1,0,World,M,100.0,"Salmon Brook St & E Granby Rd, Granby, Connect...","Salmon Brook St & E Granby Rd, Granby, CT, 060...",Salmon Brook St & E Granby Rd,StreetInt,,,...,,,,,,,,1,"{""x"": -72.789686020831, ""y"": 41.954087988452, ...","SALMON BROOK ST @ EAST GRANBY RD, GRANBY CT"
2,0,World,M,100.0,"Salmon Brook St & Hartford Ave, Granby, Connec...","Salmon Brook St & Hartford Ave, Granby, CT, 06...",Salmon Brook St & Hartford Ave,StreetInt,,,...,,,,,,,,1,"{""x"": -72.789361808817, ""y"": 41.952785817884, ...","SALMON BROOK ST @ HARTFORD AVE, GRANBY CT"
3,0,World,M,100.0,"N Granby Rd & Mountain Rd, North Granby, Conne...","N Granby Rd & Mountain Rd, North Granby, CT, 0...",N Granby Rd & Mountain Rd,StreetInt,,,...,,,,,,,,1,"{""x"": -72.830777714416, ""y"": 41.995929191818, ...","NORTH GRANBY RD @ MOUNTAIN RD, GRANBY CT"
4,0,World,T,98.14,"Silver St & Silver Brook Ln, North Granby, Con...","Silver St & Silver Brook Ln, North Granby, CT,...",Silver St & Silver Brook Ln,StreetInt,,,...,,,,,,,,1,"{""x"": -72.846929977111, ""y"": 42.035598015429, ...","SILVER ST @ SILVER BROOK LN, GRANBY CT"


In [79]:
test = pd.read_parquet('/Users/Shared/Granby_Police_Journals/geocoding/granby_intersections.parquet').reset_index(drop=False, names="ReturnNo")
       #[['Address', 'Street', 'Town', 'State', 'Status', 'Score', 'Rank', 'Addr_type', 'Postal', 'X','Y', 'ShortLabel', 'Match_addr' ]]
         #.rename(columns={'X': 'Longitude', 'Y': 'Latitude'})

rdf = rdf.merge(test[['Match_addr', 'X', 'Y', 'Address']], how='left', on=['Match_addr', 'X', 'Y'])
newres = rdf[rdf['Address'].isna()].drop(columns=['Address'])

In [80]:
newres.shape

(41, 85)

In [81]:
# verify new results are truly new
newres.merge(test[['Match_addr', 'X', 'Y']], how='inner', on=['Match_addr', 'X', 'Y'])

,ReturnNo,Loc_name,Status,Score,Match_addr,LongLabel,ShortLabel,Addr_type,Type,PlaceName,...,PotentialID,StrucType,StrucDet,StrucType1,StrucType2,StrucDet1,StrucDet2,OBJECTID,SHAPE,intersection


In [82]:
newres['Street'] = newres['intersection'].str.split(',').str[0]
newres['Town'] = newres['intersection'].str.split(',').str[1].str[:-2].str.strip()
newres['Address'] = newres.Street + ", " + newres.Town
newres['State'] = newres['intersection'].str.split(',').str[1].str[-2::].str.strip()
newres.shape, test.shape

((41, 89), (207, 89))

In [86]:
#_df = pd.concat([test, newres])
#_df.drop(columns=['SHAPE']).to_parquet('/Users/Shared/Granby_Police_Journals/geocoding/granby_intersections.parquet')

In [88]:
_df.head()

,ReturnNo,Loc_name,Status,Score,Match_addr,LongLabel,ShortLabel,Addr_type,Type,PlaceName,...,StrucType2,StrucDet1,StrucDet2,OBJECTID,SHAPE,intersection,Street,Town,Address,State
0,0,World,M,100.0,"Wells Rd & Meadowbrook Rd, Granby, Connecticut...","Wells Rd & Meadowbrook Rd, Granby, CT, 06035, USA",Wells Rd & Meadowbrook Rd,StreetInt,,,...,,,,1,b'\x01\x01\x00\x00\x00Bg\x9c\xe7\xa53R\xc0l\xf...,"WELLS RD @ MEADOW BROOK RD, GRANBY CT",WELLS RD @ MEADOW BROOK RD,GRANBY,"WELLS RD @ MEADOW BROOK RD, GRANBY",CT
1,0,World,M,100.0,"N Granby Rd & Danielle Rd, Granby, Connecticut...","N Granby Rd & Danielle Rd, Granby, CT, 06035, USA",N Granby Rd & Danielle Rd,StreetInt,,,...,,,,1,b'\x01\x01\x00\x00\x00Kf\x06\x8733R\xc0`\xf6\x...,"NORTH GRANBY RD @ DANIELLE RD, GRANBY CT",NORTH GRANBY RD @ DANIELLE RD,GRANBY,"NORTH GRANBY RD @ DANIELLE RD, GRANBY",CT
2,0,World,M,98.14,"Silver St & Doherty Rd, North Granby, Connecti...","Silver St & Doherty Rd, North Granby, CT, 0606...",Silver St & Doherty Rd,StreetInt,,,...,,,,1,b'\x01\x01\x00\x00\x00\x8ekbp\xcd5R\xc0\xff\x0...,"SILVER ST @ DOHERTY RD, GRANBY CT",SILVER ST @ DOHERTY RD,GRANBY,"SILVER ST @ DOHERTY RD, GRANBY",CT
3,0,World,M,100.0,"E Granby Rd & Laurel Dr, Granby, Connecticut, ...","E Granby Rd & Laurel Dr, Granby, CT, 06035, USA",E Granby Rd & Laurel Dr,StreetInt,,,...,,,,1,b'\x01\x01\x00\x00\x00\x85c\xc6\x14\xba1R\xc0\...,"EAST GRANBY RD @ LAUREL DR, GRANBY CT",EAST GRANBY RD @ LAUREL DR,GRANBY,"EAST GRANBY RD @ LAUREL DR, GRANBY",CT
4,0,World,M,98.14,"Granville Rd & Northwoods Rd, North Granby, Co...","Granville Rd & Northwoods Rd, North Granby, CT...",Granville Rd & Northwoods Rd,StreetInt,,,...,,,,1,b'\x01\x01\x00\x00\x00\x89n`><7R\xc0\xe1\x08\x...,"GRANVILLE RD @ NORTHWOODS RD, GRANBY CT",GRANVILLE RD @ NORTHWOODS RD,GRANBY,"GRANVILLE RD @ NORTHWOODS RD, GRANBY",CT


In [89]:
#_df[['ReturnNo', 'Address', 'Street', 'Town', 'State', 'Status', 'Score', 'Rank', 'Addr_type', 'Postal', 'X','Y', 'ShortLabel', 'Match_addr' ]]\
#    .rename(columns={'X': 'Longitude', 'Y': 'Latitude'}).to_csv('/Users/Shared/Granby_Police_Journals/geocoding/granby_intersections_geocoded.csv')

In [36]:
#[['Address', 'Street', 'Town', 'State', 'Status', 'Score', 'Rank', 'Addr_type', 'Postal', 'X','Y', 'ShortLabel', 'Match_addr' ]]
#arcgeo = pd.read_csv('/Users/Shared/Granby_Police_Journals/geocoding/granby_intersections_geocoded.csv')\
#           .rename(columns={'X': 'Longitude', 'Y': 'Latitude'})

In [44]:
arcgeo.drop_duplicates(subset='Address').shape

(137, 14)

In [50]:
arcgeo[arcgeo.ResultNo == 0]['Addr_type'].value_counts()

Addr_type
StreetInt     132
StreetName      3
Locality        2
Name: count, dtype: int64

In [60]:
arcgeo.columns

Index(['ResultNo', 'Address', 'Street', 'Town', 'State', 'Status', 'Score',
       'Rank', 'Addr_type', 'Postal', 'Longitude', 'Latitude', 'ShortLabel',
       'Match_addr'],
      dtype='str')

In [62]:
origcols = [c for c in df.columns if c in df4.columns]
df4[origcols].merge(arcgeo[arcgeo.ResultNo == 0][['Address', 'Town', 'State', 'Postal', 'Longitude', 'Latitude']],
                    how='left', on=['Address', 'Town', 'State'])

,CFS #,Dispatch Type,Call Type,Dispatch Date,Dispatch Time,Location,Unit Id,Primary Officer,File Date,Dispatch Hour,...,Dispatch Night,Place Name,Address,Town,State,Street,Street Name,Postal,Longitude,Latitude
0,2500007994,17,MV COMPLAINT,2025-07-14,13:29,"WELLS RD/MEADOW BROOK RD, GRANBY",104,"MIKAN, DOREEN",07-14-2025,13,...,2025-07-14,,"WELLS RD @ MEADOW BROOK RD, GRANBY",GRANBY,CT,WELLS RD @ MEADOW BROOK RD,WELLS RD @ MEADOW BROOK RD,6035.0,-72.807001,41.982008
1,2500007999,99,911 UNKNOWN,2025-07-14,16:16,CELL TOWER,999,,07-14-2025,16,...,2025-07-14,,CELL TOWER,CELL TOWER,,CELL TOWER,CELL TOWER,NaN,NaN,NaN
2,2500008021,17,PUBLIC HAZARD,2025-07-15,09:21,"NORTH GRANBY RD/DANIELLE RD, GRANBY",265,"DUFRESNE, CHRISTOP",07-15-2025,9,...,2025-07-15,,"NORTH GRANBY RD @ DANIELLE RD, GRANBY",GRANBY,CT,NORTH GRANBY RD @ DANIELLE RD,NORTH GRANBY RD @ DANIELLE RD,6035.0,-72.800020,41.961681
3,2500008032,21,MV STOP,2025-07-15,15:44,"SILVER ST/DOHERTY RD, GRANBY",254,"WALZAK, M. PAUL",07-15-2025,15,...,2025-07-15,,"SILVER ST @ DOHERTY RD, GRANBY",GRANBY,CT,SILVER ST @ DOHERTY RD,SILVER ST @ DOHERTY RD,6060.0,-72.840664,42.019834
4,2500008064,,,2025-07-16,12:52,"EAST GRANBY RD/LAUREL DR, GRANBY",264,"DZIERZGOWSKI, ANDR",07-16-2025,12,...,2025-07-16,,"EAST GRANBY RD @ LAUREL DR, GRANBY",GRANBY,CT,EAST GRANBY RD @ LAUREL DR,EAST GRANBY RD @ LAUREL DR,6035.0,-72.776982,41.954710
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
304,2500014671,21,MV STOP,2025-12-17,21:26,"EAST GRANBY RD/BANK ST, GRANBY",267,"WILKINS, JAMES",12-17-2025,21,...,2025-12-17,,"EAST GRANBY RD @ BANK ST, GRANBY",GRANBY,CT,EAST GRANBY RD @ BANK ST,EAST GRANBY RD @ BANK ST,6035.0,-72.785017,41.953637
305,2500014684,21,MV STOP,2025-12-18,01:40,"EAST GRANBY RD/OAKRIDGE DR, GRANBY",262,"MARTINAJ, RAM",12-18-2025,1,...,2025-12-18,,"EAST GRANBY RD @ OAKRIDGE DR, GRANBY",GRANBY,CT,EAST GRANBY RD @ OAKRIDGE DR,EAST GRANBY RD @ OAKRIDGE DR,6035.0,-72.782888,41.953536
306,2500014699,18,MV ACCIDENT - PERSONAL INJURY,2025-12-18,10:25,"CANTON RD/SHELLEY DR, GRANBY",264,"DZIERZGOWSKI, ANDR",12-18-2025,10,...,2025-12-18,,"CANTON RD @ SHELLEY DR, GRANBY",GRANBY,CT,CANTON RD @ SHELLEY DR,CANTON RD @ SHELLEY DR,6035.0,-72.802625,41.929278
307,2500014706,17,SELECTIVE ENFORCEMENT,2025-12-18,15:47,"MEADOW BROOK RD/SUSAN LN, GRANBY",267,"WILKINS, JAMES",12-18-2025,15,...,2025-12-18,,"MEADOW BROOK RD @ SUSAN LN, GRANBY",GRANBY,CT,MEADOW BROOK RD @ SUSAN LN,MEADOW BROOK RD @ SUSAN LN,6035.0,-72.803514,41.984953


In [27]:
pd.read_parquet('/Users/Shared/Granby_Police_Journals/geocoding/granby_intersections.parquet')[['Address', 'Street', 'Town', 'State', 'Status', 'Score', 'Rank', 'Addr_type', 'Postal', 'X','Y', 'ShortLabel', 'Match_addr' ]]\
         .rename(columns={'X': 'Longitude', 'Y': 'Latitude'})

,Address,Street,Town,State,Status,Score,Rank,Addr_type,Postal,Longitude,Latitude,ShortLabel,Match_addr
0,"WELLS RD @ MEADOW BROOK RD, GRANBY",WELLS RD @ MEADOW BROOK RD,GRANBY,CT,M,100.0,20.0,StreetInt,06035,-72.807001,41.982008,Wells Rd & Meadowbrook Rd,"Wells Rd & Meadowbrook Rd, Granby, Connecticut..."
0,"NORTH GRANBY RD @ DANIELLE RD, GRANBY",NORTH GRANBY RD @ DANIELLE RD,GRANBY,CT,M,100.0,20.0,StreetInt,06035,-72.80002,41.961681,N Granby Rd & Danielle Rd,"N Granby Rd & Danielle Rd, Granby, Connecticut..."
0,"SILVER ST @ DOHERTY RD, GRANBY",SILVER ST @ DOHERTY RD,GRANBY,CT,M,98.14,20.0,StreetInt,06060,-72.840664,42.019834,Silver St & Doherty Rd,"Silver St & Doherty Rd, North Granby, Connecti..."
0,"EAST GRANBY RD @ LAUREL DR, GRANBY",EAST GRANBY RD @ LAUREL DR,GRANBY,CT,M,100.0,20.0,StreetInt,06035,-72.776982,41.95471,E Granby Rd & Laurel Dr,"E Granby Rd & Laurel Dr, Granby, Connecticut, ..."
0,"GRANVILLE RD @ NORTHWOODS RD, GRANBY",GRANVILLE RD @ NORTHWOODS RD,GRANBY,CT,M,98.14,20.0,StreetInt,06060,-72.863052,42.033944,Granville Rd & Northwoods Rd,"Granville Rd & Northwoods Rd, North Granby, Co..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,"CANAL RD @ HUNGARY RD, GRANBY",CANAL RD @ HUNGARY RD,GRANBY,CT,M,100.0,20.0,StreetInt,06035,-72.783665,41.95881,Canal Rd & Hungary Rd,"Canal Rd & Hungary Rd, Granby, Connecticut, 06035"
0,"EAST ST @ COOLEY RD, GRANBY",EAST ST @ COOLEY RD,GRANBY,CT,M,100.0,20.0,StreetInt,06035,-72.817131,41.99749,East St & Cooley Rd,"East St & Cooley Rd, Granby, Connecticut, 06035"
0,"SILVER ST @ STONEHEDGE WAY, GRANBY",SILVER ST @ STONEHEDGE WAY,GRANBY,CT,T,98.14,20.0,StreetInt,06060,-72.844049,42.02465,Silver St & Stonehedge Way,"Silver St & Stonehedge Way, North Granby, Conn..."
1,"SILVER ST @ STONEHEDGE WAY, GRANBY",SILVER ST @ STONEHEDGE WAY,GRANBY,CT,T,98.14,20.0,StreetInt,06060,-72.840664,42.019834,Silver St & Stonehedge Way,"Silver St & Stonehedge Way, North Granby, Conn..."


In [32]:
#with pd.option_context("display.max_rows", None, 'display.max_colwidth', 100):
g = pd.DataFrame(df[(df['lat'].notna()) & (~df['Call Type'].isin(['POLICE INFO', 'SCHOOL TRAFFIC'])) & (df['Call Category'] != 'Administrative') ][['Call Category', 'Call Type', 'lat', 'lon']].value_counts())
g.reset_index(inplace=True)
g.rename(columns={"Call Category": "Category", "Call Type": "Call", 'lat': 'Latitude', 'lon': 'Longitude', 'count':'Count'}, inplace=True)
g['Call'] = g['Call'].str.title()
g.to_csv("/Users/joel/Desktop/call_counts.csv", header=True, index=False)

### Retrieve geocoded information

In [27]:
# add locations
df = df.merge(geocodes[['Address', 'ZIP Code', 'lat', 'lon']].drop_duplicates(subset=['Address']), how='left', on='Address')
df['ZIP Code'] = df['ZIP Code'].fillna('00000')

In [28]:
ziplooks = df[(df['Town'] == 'GRANBY') & (df['ZIP Code'] != '00000') & (~df['Address'].str.contains('@'))][['Street', 'ZIP Code']].drop_duplicates(subset='Street')
zipdict = dict(zip(ziplooks.Street.values, ziplooks['ZIP Code'].values))

In [30]:
# fill in zips for more matching
df['ZIP Code'] = df['ZIP Code'].where(df['ZIP Code'] != '00000', df['Address'].apply(lambda x:zipdict.get(x.split('@')[0].strip(), '00000')))
df['ZIP Code'] = df['ZIP Code'].where(df['ZIP Code'] != '00000', df['Address'].apply(lambda x:zipdict.get(x.split('@')[::-1][0].split(',')[0].strip(), '00000')))
df['ZIP Code'] = df['ZIP Code'].where(df['ZIP Code'] != '00000', df['Town'].apply(lambda x: {'GRANBY': '06035'}.get(x, '00000')))

In [33]:
# write another file - pass to Census geocoder https://geocoding.geo.census.gov/geocoder/locations/addressbatch?form

g = df[(df['Address'].str.count(",") == 1) & (df['State'].str.len() > 1)][['Address', 'Town', 'State', 'ZIP Code']]
g['Address'] = g['Address'].str.split(",").str[0]
g.drop_duplicates(inplace=True)
g.shape
#g.to_csv("/Users/joel/Desktop/addresses_with_zips.csv", header=False)

(1724, 4)

In [34]:
censusmatch = pd.read_csv("/Users/joel/Desktop/CensusGeocodeResults.csv")
censusmatch[censusmatch['match'] == 'No_Match']

,index,address,match,type,clean_address,latlon,tigerline,side
0,1462,"MAPLE HILL DR, GRANBY, CT, 06035",No_Match,NaN,NaN,NaN,NaN,NaN
18,5,"BARKHAMSTED RD, GRANBY, CT, 06090",No_Match,NaN,NaN,NaN,NaN,NaN
24,5822,"RAINBOW RD @ SCHOOL ST, EAST GRANBY, CT, 00000",No_Match,NaN,NaN,NaN,NaN,NaN
28,4992,"WINDY WOODS LN, GRANBY, CT, 06090",No_Match,NaN,NaN,NaN,NaN,NaN
35,2322,"MORNINGSIDE DR, GRANBY, CT, 06090",No_Match,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
1682,2732,"FARMVIEW LN, GRANBY, CT, 06035",No_Match,NaN,NaN,NaN,NaN,NaN
1685,2745,"DAY ST SOUTH, GRANBY, CT, 06090",No_Match,NaN,NaN,NaN,NaN,NaN
1695,3600,"HUMMINGBIRD LN, GRANBY, CT, 06035",No_Match,NaN,NaN,NaN,NaN,NaN
1703,1433,"PARTRIDGE MDW, GRANBY, CT, 06035",No_Match,NaN,NaN,NaN,NaN,NaN


In [127]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter as RateLimiter

# Specify a custom user_agent to comply with Nominatim's usage policy
geolocator = Nominatim(user_agent="police_log_lookup/1.0 (joel.danke@gmail.com)")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

address = "NORTH GRANBY RD @ BUSHY HILL RD, GRANBY CT"

location = geocode(address)

if location:
    print(f"Address: {location.address}")
    print(f"Latitude: {location.latitude}")
    print(f"Longitude: {location.longitude}")
else:
    print("Location not found.")

Location not found.


In [ ]:
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# Example geocoding call
location = geocode("Portland, Oregon")
if location:
    print(f"The location is at: {location.latitude}, {location.longitude}")
else:
    print("Location not found.")

In [18]:
# Connect to ArcGIS Online anonymously
# A named user connection might be required for production use
gis = GIS() 

# Example of an intersection query
intersection_query = "NORTH GRANBY RD @ BUSHY HILL RD, GRANBY CT"

# Perform geocoding
try:
    results = geocode(address=intersection_query, as_featureset=True)
    if results:
        # The best match is typically the first result
        location = results[0]['location']
        print(f"Intersection Coordinates: Latitude: {location['y']}, Longitude: {location['x']}")
    else:
        print("No intersection found.")
except Exception as e:
    print(f"An error occurred: {e}")

An error occurred: 'FeatureSet' object is not subscriptable


In [24]:
results.sdf.columns

Index(['Loc_name', 'Status', 'Score', 'Match_addr', 'LongLabel', 'ShortLabel',
       'Addr_type', 'Type', 'PlaceName', 'Place_addr', 'Phone', 'URL', 'Rank',
       'AddBldg', 'AddNum', 'AddNumFrom', 'AddNumTo', 'AddRange', 'Side',
       'StPreDir', 'StPreType', 'StName', 'StType', 'StDir', 'StPreDir1',
       'StPreType1', 'StName1', 'StType1', 'StDir1', 'StPreDir2', 'StPreType2',
       'StName2', 'StType2', 'StDir2', 'BldgComp', 'BldgType', 'BldgName',
       'LevelType', 'LevelName', 'UnitType', 'UnitName', 'RoomType',
       'RoomName', 'WingType', 'WingName', 'SubAddr', 'StAddr', 'Block',
       'Sector', 'Nbrhd', 'District', 'City', 'MetroArea', 'Subregion',
       'Region', 'RegionAbbr', 'Territory', 'Zone', 'Postal', 'PostalExt',
       'Country', 'CntryName', 'LangCode', 'Distance', 'X', 'Y', 'DisplayX',
       'DisplayY', 'Xmin', 'Xmax', 'Ymin', 'Ymax', 'ExInfo', 'MatchID',
       'PotentialID', 'StrucType', 'StrucDet', 'StrucType1', 'StrucType2',
       'StrucDet1', 'Struc

In [9]:
intersections = pd.read_csv('/Users/Shared/Granby_Police_Journals/geocoding/granby_intersections.csv')
intersections['address'] = intersections.Street + ", " + intersections.Town + " " + intersections.State

In [ ]:
intersections['address'].tolist()

In [87]:
# Connect to ArcGIS Online anonymously
# A named user connection might be required for production use
gis = GIS() 

georesults = []
#for intersection_query in intersections['address'].tolist():
for intersection_query in g['intersection'].tolist():

    time.sleep(2)
    # Perform geocoding
    print(f"{intersection_query}")
    try:
        results = geocode(address=intersection_query, as_featureset=True)
        if results:
            geodf = results.sdf
            geodf['intersection'] = intersection_query
            georesults.append(geodf)
        else:
            print(f"      --->{intersection_query}: No intersection found.")
    except Exception as e:
        print(f"An error occurred: {e}")

NOTCH RD @ HUNGARY RD, GRANBY CT
An error occurred: An error occurred with exporting the FeatureSet with message: could not convert string to float: '0rc1'
SALMON BROOK ST @ EAST GRANBY RD, GRANBY CT
An error occurred: An error occurred with exporting the FeatureSet with message: could not convert string to float: '0rc1'
SALMON BROOK ST @ HARTFORD AVE, GRANBY CT
An error occurred: An error occurred with exporting the FeatureSet with message: could not convert string to float: '0rc1'
NORTH GRANBY RD @ MOUNTAIN RD, GRANBY CT
An error occurred: An error occurred with exporting the FeatureSet with message: could not convert string to float: '0rc1'
SILVER ST @ SILVER BROOK LN, GRANBY CT
An error occurred: An error occurred with exporting the FeatureSet with message: could not convert string to float: '0rc1'
GRANBROOK PARK RD @ HARTFORD AVE, EAST GRANBY CT
An error occurred: An error occurred with exporting the FeatureSet with message: could not convert string to float: '0rc1'


KeyboardInterrupt: 

In [44]:
arcgisdf = pd.concat(georesults)
arcgisdf['Street'] = arcgisdf['intersection'].str.split(',').str[0]
arcgisdf['Town'] = arcgisdf['intersection'].str.split(',').str[1].str.split(' ').str[1]
arcgisdf['Address'] = arcgisdf.Street + ", " + arcgisdf.Town
arcgisdf['State'] = arcgisdf['intersection'].str.split(',').str[1].str.split(' ').str[2]

In [45]:
#arcgisdf.to_parquet('/Users/Shared/Granby_Police_Journals/geocoding/granby_intersections.parquet')

In [72]:
#arcgisdf[['Address', 'Street', 'Town', 'State', 'Status', 'Score', 'Rank', 'Addr_type', 'Postal', 'X','Y', 'ShortLabel', 'Match_addr' ]]\
#         .rename(columns={'X': 'Longitude', 'Y': 'Latitude'}).to_csv('/Users/Shared/Granby_Police_Journals/geocoding/granby_intersections_geocoded.csv')

,Address,Street,Town,State,Status,Score,Rank,Addr_type,Postal,Longitude,Latitude,ShortLabel,Match_addr
0,"WELLS RD @ MEADOW BROOK RD, GRANBY",WELLS RD @ MEADOW BROOK RD,GRANBY,CT,M,100.0,20.0,StreetInt,06035,-72.807001,41.982008,Wells Rd & Meadowbrook Rd,"Wells Rd & Meadowbrook Rd, Granby, Connecticut..."
0,"NORTH GRANBY RD @ DANIELLE RD, GRANBY",NORTH GRANBY RD @ DANIELLE RD,GRANBY,CT,M,100.0,20.0,StreetInt,06035,-72.80002,41.961681,N Granby Rd & Danielle Rd,"N Granby Rd & Danielle Rd, Granby, Connecticut..."
0,"SILVER ST @ DOHERTY RD, GRANBY",SILVER ST @ DOHERTY RD,GRANBY,CT,M,98.14,20.0,StreetInt,06060,-72.840664,42.019834,Silver St & Doherty Rd,"Silver St & Doherty Rd, North Granby, Connecti..."
0,"EAST GRANBY RD @ LAUREL DR, GRANBY",EAST GRANBY RD @ LAUREL DR,GRANBY,CT,M,100.0,20.0,StreetInt,06035,-72.776982,41.95471,E Granby Rd & Laurel Dr,"E Granby Rd & Laurel Dr, Granby, Connecticut, ..."
0,"GRANVILLE RD @ NORTHWOODS RD, GRANBY",GRANVILLE RD @ NORTHWOODS RD,GRANBY,CT,M,98.14,20.0,StreetInt,06060,-72.863052,42.033944,Granville Rd & Northwoods Rd,"Granville Rd & Northwoods Rd, North Granby, Co..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,"CANAL RD @ HUNGARY RD, GRANBY",CANAL RD @ HUNGARY RD,GRANBY,CT,M,100.0,20.0,StreetInt,06035,-72.783665,41.95881,Canal Rd & Hungary Rd,"Canal Rd & Hungary Rd, Granby, Connecticut, 06035"
0,"EAST ST @ COOLEY RD, GRANBY",EAST ST @ COOLEY RD,GRANBY,CT,M,100.0,20.0,StreetInt,06035,-72.817131,41.99749,East St & Cooley Rd,"East St & Cooley Rd, Granby, Connecticut, 06035"
0,"SILVER ST @ STONEHEDGE WAY, GRANBY",SILVER ST @ STONEHEDGE WAY,GRANBY,CT,T,98.14,20.0,StreetInt,06060,-72.844049,42.02465,Silver St & Stonehedge Way,"Silver St & Stonehedge Way, North Granby, Conn..."
1,"SILVER ST @ STONEHEDGE WAY, GRANBY",SILVER ST @ STONEHEDGE WAY,GRANBY,CT,T,98.14,20.0,StreetInt,06060,-72.840664,42.019834,Silver St & Stonehedge Way,"Silver St & Stonehedge Way, North Granby, Conn..."
